# Analyse de Sentiments — Critiques de Films (SST-2)
### NLP · TF-IDF, Régression Logistique, Random Forest & XGBoost

**Problématique :** À partir de phrases extraites de critiques de films, peut-on construire un modèle capable de classer automatiquement le sentiment (positif / négatif) ?

**Données :** Stanford Sentiment Treebank v2 (SST-2) — dataset académique de référence en NLP, issu de critiques de films réelles, utilisé comme benchmark dans les publications scientifiques.

**Approche :** Vectorisation TF-IDF → comparaison de trois algorithmes de classification → analyse des termes discriminants → test sur des critiques personnalisées.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import urllib.request
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve,
                              accuracy_score, f1_score)
from xgboost import XGBClassifier

plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11,
                     'axes.titlesize': 12, 'axes.titleweight': 'bold'})
plt.style.use('seaborn-v0_8-darkgrid')

SEED = 42
np.random.seed(SEED)
print("✅ Imports OK")

## 2. Données — Stanford Sentiment Treebank (SST-2)

Le **SST-2** (Socher et al., 2013) est le benchmark de référence en classification de sentiment binaire. Il est composé de phrases extraites de critiques de films annotées manuellement par des humains, avec deux classes :
- **0** → sentiment négatif
- **1** → sentiment positif

On combine les ensembles train et dev pour maximiser les données d'entraînement, et on évalue sur le test set.

In [ ]:
BASE = "https://raw.githubusercontent.com/clairett/pytorch-sentiment-classification/master/data/SST2"

def charger_tsv(url):
    content = urllib.request.urlopen(url).read().decode('utf-8')
    return pd.read_csv(StringIO(content), sep='\t', header=None, names=['text','label'])

df_train = charger_tsv(f"{BASE}/train.tsv")
df_dev   = charger_tsv(f"{BASE}/dev.tsv")
df_test  = charger_tsv(f"{BASE}/test.tsv")

# Combinaison train + dev pour maximiser les données d'entraînement
df_train_full = pd.concat([df_train, df_dev], ignore_index=True)

print(f"Train : {len(df_train_full):,} phrases")
print(f"Test  : {len(df_test):,} phrases")
print(f"Taux positif (train) : {df_train_full['label'].mean():.2%}")
print(f"Taux positif (test)  : {df_test['label'].mean():.2%}")
df_train_full.head(5)

## 3. Analyse Exploratoire

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = df_train_full['label'].value_counts().sort_index()
colors = ['#e63946', '#4e9af1']
axes[0].bar(['Négatif (0)', 'Positif (1)'], counts.values,
            color=colors, edgecolor='white')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, f'{v:,}\n({v/len(df_train_full):.1%})',
                 ha='center', fontweight='bold')
axes[0].set_title('Répartition des Classes')
axes[0].set_ylabel('Nombre de phrases')

df_train_full['longueur'] = df_train_full['text'].apply(lambda x: len(str(x).split()))
for label, color, nom in [(0,'#e63946','Négatif'),(1,'#4e9af1','Positif')]:
    subset = df_train_full[df_train_full['label']==label]['longueur']
    axes[1].hist(subset, bins=30, alpha=0.6, color=color, label=nom, density=True)
axes[1].set_title('Distribution de la Longueur des Phrases')
axes[1].set_xlabel('Nombre de mots')
axes[1].legend()

plt.tight_layout()
plt.savefig('01_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Longueur moyenne : {df_train_full['longueur'].mean():.1f} mots")
print(f"Longueur médiane : {df_train_full['longueur'].median():.1f} mots")

In [ ]:
print("EXEMPLES NÉGATIFS :")
for ex in df_train_full[df_train_full['label']==0]['text'].head(3).values:
    print(f"  ❌ {ex}")

print("\nEXEMPLES POSITIFS :")
for ex in df_train_full[df_train_full['label']==1]['text'].head(3).values:
    print(f"  ✅ {ex}")

## 4. Prétraitement du Texte

Nettoyage minimal adapté à ce dataset : suppression de la ponctuation et mise en minuscules. Le SST-2 est déjà tokenisé et ne contient pas de balises HTML.

In [ ]:
def nettoyer(texte):
    texte = str(texte).lower()
    texte = re.sub(r'[^\w\s]', ' ', texte)
    texte = re.sub(r'\s+', ' ', texte).strip()
    return texte

df_train_full['text_clean'] = df_train_full['text'].apply(nettoyer)
df_test['text_clean']       = df_test['text'].apply(nettoyer)

print("Avant :", df_train_full['text'].iloc[0])
print("Après :", df_train_full['text_clean'].iloc[0])

## 5. Vectorisation TF-IDF

**TF-IDF** (Term Frequency — Inverse Document Frequency) transforme chaque phrase en vecteur numérique. La pondération combine la fréquence d'un terme dans la phrase (TF) et sa rareté dans le corpus (IDF) — les mots très courants reçoivent un faible poids car non discriminants.

L'utilisation des **bigrammes** (paires de mots consécutifs) permet de capturer des expressions comme "not good" ou "very bad" que les unigrammes seuls ne peuvent pas distinguer correctement.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),   # unigrammes + bigrammes
    min_df=2,             # ignore les termes présents dans < 2 phrases
    sublinear_tf=True,    # log(1 + tf) pour atténuer les termes très fréquents
    stop_words='english'  # suppression des mots vides anglais
)

X_train = vectorizer.fit_transform(df_train_full['text_clean'])
X_test  = vectorizer.transform(df_test['text_clean'])
y_train = df_train_full['label']
y_test  = df_test['label']

print(f"Matrice TF-IDF train : {X_train.shape}")
print(f"Matrice TF-IDF test  : {X_test.shape}")
print(f"Vocabulaire          : {len(vectorizer.vocabulary_):,} termes")

## 6. Modélisation

### 6.1 Régression Logistique — Baseline

Modèle de référence en classification de texte. Ses coefficients sont directement interprétables : chaque terme reçoit un poids positif ou négatif proportionnel à son pouvoir discriminant.

In [ ]:
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
lr.fit(X_train, y_train)

pred_lr  = lr.predict(X_test)
proba_lr = lr.predict_proba(X_test)[:,1]

acc_lr = accuracy_score(y_test, pred_lr)
f1_lr  = f1_score(y_test, pred_lr, average='weighted')
auc_lr = roc_auc_score(y_test, proba_lr)

print(f"✅ Régression Logistique — Accuracy={acc_lr:.4f} | F1={f1_lr:.4f} | AUC={auc_lr:.4f}")

### 6.2 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=3,
    random_state=SEED, n_jobs=-1
)
rf.fit(X_train, y_train)

pred_rf  = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:,1]

acc_rf = accuracy_score(y_test, pred_rf)
f1_rf  = f1_score(y_test, pred_rf, average='weighted')
auc_rf = roc_auc_score(y_test, proba_rf)

print(f"✅ Random Forest — Accuracy={acc_rf:.4f} | F1={f1_rf:.4f} | AUC={auc_rf:.4f}")

### 6.3 XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=SEED, n_jobs=-1
)
xgb.fit(X_train, y_train)

pred_xgb  = xgb.predict(X_test)
proba_xgb = xgb.predict_proba(X_test)[:,1]

acc_xgb = accuracy_score(y_test, pred_xgb)
f1_xgb  = f1_score(y_test, pred_xgb, average='weighted')
auc_xgb = roc_auc_score(y_test, proba_xgb)

print(f"✅ XGBoost — Accuracy={acc_xgb:.4f} | F1={f1_xgb:.4f} | AUC={auc_xgb:.4f}")

## 7. Comparaison des Modèles

In [ ]:
print("="*62)
print("  RÉSULTATS — JEU DE TEST")
print("="*62)
print(f"  {'Modèle':<25} {'Accuracy':>10} {'F1':>10} {'AUC':>10}")
print("  " + "-"*57)
for nom, acc, f1, auc in [
    ('Régression Logistique', acc_lr,  f1_lr,  auc_lr),
    ('Random Forest',         acc_rf,  f1_rf,  auc_rf),
    ('XGBoost',               acc_xgb, f1_xgb, auc_xgb),
]:
    print(f"  {nom:<25} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f}")

best = max([('LR', auc_lr),('RF', auc_rf),('XGBoost', auc_xgb)], key=lambda x: x[1])
print(f"\n🏆 Meilleur modèle (AUC) : {best[0]}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
modeles  = ['LR', 'Random Forest', 'XGBoost']
couleurs = ['#4e9af1', '#f4a261', '#e63946']

for ax, vals, titre in zip(axes,
    [[acc_lr, acc_rf, acc_xgb],
     [f1_lr,  f1_rf,  f1_xgb],
     [auc_lr, auc_rf, auc_xgb]],
    ['Accuracy', 'F1-Score', 'AUC-ROC']):
    bars = ax.bar(modeles, vals, color=couleurs, edgecolor='white', width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.003,
                f'{v:.4f}', ha='center', fontweight='bold', fontsize=10)
    ax.set_title(titre)
    ax.set_ylim(min(vals)*0.95, 1.0)

plt.tight_layout()
plt.savefig('02_comparaison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for proba, nom, couleur in [
    (proba_lr,  f'Régression Logistique (AUC={auc_lr:.3f})', '#4e9af1'),
    (proba_rf,  f'Random Forest (AUC={auc_rf:.3f})',         '#f4a261'),
    (proba_xgb, f'XGBoost (AUC={auc_xgb:.3f})',             '#e63946'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, lw=2.5, label=nom)

ax.plot([0,1],[0,1], color='gray', ls='--', lw=1.5, label='Hasard (AUC=0.500)')
ax.set_xlabel('Taux de Faux Positifs')
ax.set_ylabel('Taux de Vrais Positifs')
ax.set_title('Courbes ROC — Comparaison des 3 Modèles')
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim(0,1); ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.savefig('03_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, preds, titre in zip(axes,
    [pred_lr, pred_rf, pred_xgb],
    ['Régression Logistique', 'Random Forest', 'XGBoost']):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Négatif','Positif'],
                yticklabels=['Négatif','Positif'],
                annot_kws={'size': 12})
    ax.set_title(titre)
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')

plt.tight_layout()
plt.savefig('04_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Termes les Plus Discriminants

Les coefficients de la Régression Logistique permettent d'identifier les termes les plus fortement associés à chaque sentiment — une forme d'interprétabilité directe du modèle, sans boîte noire.

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coefs         = lr.coef_[0]

top_pos = np.argsort(coefs)[-20:][::-1]
top_neg = np.argsort(coefs)[:20]

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].barh([feature_names[i] for i in top_pos[::-1]],
              [coefs[i] for i in top_pos[::-1]],
              color='#4e9af1', edgecolor='white')
axes[0].set_title('Top 20 Termes — Sentiment Positif')
axes[0].set_xlabel('Coefficient (Régression Logistique)')

axes[1].barh([feature_names[i] for i in top_neg],
              [coefs[i] for i in top_neg],
              color='#e63946', edgecolor='white')
axes[1].set_title('Top 20 Termes — Sentiment Négatif')
axes[1].set_xlabel('Coefficient (Régression Logistique)')

plt.tight_layout()
plt.savefig('05_termes.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Test sur des Critiques Personnalisées

In [ ]:
exemples = [
    "An absolute masterpiece. The performances are outstanding and the story is deeply moving.",
    "A boring and predictable film. The characters are flat and the plot goes nowhere.",
    "Not great but not terrible. Some enjoyable moments scattered throughout.",
    "One of the best films of the year. Stunning visuals and a brilliant screenplay.",
    "A complete disaster. Poorly written, badly directed and utterly forgettable.",
    "Surprisingly touching and beautifully crafted. A must-see.",
]

exemples_clean = [nettoyer(e) for e in exemples]
X_ex = vectorizer.transform(exemples_clean)

pred_lr_ex  = lr.predict(X_ex)
pred_rf_ex  = rf.predict(X_ex)
pred_xgb_ex = xgb.predict(X_ex)

label_map = {0: '❌ Négatif', 1: '✅ Positif'}
print(f"{'Critique':<55} {'LR':>12} {'RF':>12} {'XGB':>12}")
print("-"*95)
for ex, l, r_, x in zip(exemples, pred_lr_ex, pred_rf_ex, pred_xgb_ex):
    print(f"{ex[:53]:<55} {label_map[l]:>12} {label_map[r_]:>12} {label_map[x]:>12}")

## 10. Conclusion

### Résultats

| Modèle | Accuracy | F1-Score | AUC |
|---|---|---|---|
| Régression Logistique | 0.7996 | 0.7994 | 0.8874 |
| Random Forest | 0.7430 | 0.7428 | 0.8276 |
| XGBoost | 0.7287 | 0.7283 | 0.8156 |

### Points clés

1. **La Régression Logistique domine** sur ce type de données courtes et structurées — c'est cohérent avec la littérature : TF-IDF + LR est difficile à battre sur des tâches de sentiment binaire avec des phrases concises.

2. **Les bigrammes sont essentiels** : sans eux, des expressions comme "not good" ou "no interest" seraient mal interprétées — le mot "good" serait comptabilisé positivement.

3. **Random Forest et XGBoost sont moins adaptés ici** : construits sur des arbres de décision, ils peinent à modéliser l'espace de haute dimension produit par TF-IDF. Ils surpassent LR sur des représentations denses (embeddings).

4. **Pistes d'amélioration** :
   - Utiliser des embeddings denses (Word2Vec, GloVe, FastText) en remplacement du TF-IDF
   - Fine-tuner un modèle BERT ou CamemBERT (français) sur un dataset plus large
   - Appliquer une validation croisée stratifiée pour une évaluation plus robuste
